# Verify one ZM channel — start with FIR only

Run one cell at a time and stop at **Stop here: review FIR**. No AcqKnowledge comparison has been run yet.

For this first test, save a **separate copy** of one recording with its unchanged **raw ZM** and the **FIR result**. Use FIR 28–500 Hz, Blackman −61 dB, optimized coefficient count, and the entire waveform. Do not crop, resample or change units. Comb and RMS channels are not needed yet.

We compare the original raw samples with the raw channel retained in that copy, then compare Python FIR against AcqKnowledge FIR, sample for sample. Sample zero must represent the same instant. Errors use the original channel's units; sample indices are zero-based.

The function below is a standalone snapshot from `notebook.ipynb`; its TODOs remain unresolved. No acceptance threshold is assumed and no files are written. Future comb/RMS cells remain below the FIR stopping point.

In [2]:
from pathlib import Path
import bioread
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

## 1. Read the original and AcqKnowledge reference
Replace the two paths, then run the loading cell.

In [3]:
reference_file = Path("TEST_fEMG_div_4SlowRate_raw.acq")

In [4]:
reference_recording = bioread.read(reference_file)

In [5]:
pd.DataFrame([
    {"file": label, "index": i, "name": ch.name, "units": ch.units,
    "rate_hz": ch.samples_per_second, "samples": len(ch.data)}
    for label, recording in [("reference", reference_recording)]
    for i, ch in enumerate(recording.channels)
])

,file,index,name,units,rate_hz,samples
0,reference,0,ZM - FIR,mV,2000.0,805412
1,reference,1,ZM - Comb,mV,2000.0,805412
2,reference,2,ZM - EMG100C,mV,2000.0,805412
3,reference,3,CS - EMG100C,mV,2000.0,805412
4,reference,4,ZM - Integrate,mV,2000.0,805412


Choose the **original raw ZM index**, then the **raw ZM and FIR indices in the reference file**. Use the Python `index` column above. Only these three choices are needed for FIR.

In [6]:
raw_index = 2
reference_indices_fir = 0
reference_indices_comb = 1
reference_indices_integrate = 4

In [7]:
raw_channel = reference_recording.channels[raw_index]
reference_channel_fir = reference_recording.channels[reference_indices_fir]

reference_channel_comb = reference_recording.channels[reference_indices_comb]
reference_channel_integrate = reference_recording.channels[reference_indices_integrate]


In [8]:
raw_channel

Channel ZM - EMG100C: 805412 samples, 2000.0 samples/sec, loaded: True

In [9]:
reference_channel_fir

Channel ZM - FIR: 805412 samples, 2000.0 samples/sec, loaded: True

In [10]:
def compare(actual, expected, sampling_rate):
    actual = np.asarray(actual, dtype=float)
    expected = np.asarray(expected, dtype=float)

    if (
        actual.ndim != 1
        or expected.ndim != 1
        or actual.shape != expected.shape
        or actual.size == 0
    ):
        raise ValueError("Signals must be 1D with the same nonzero length.")

    if not (np.isfinite(actual).all() and np.isfinite(expected).all()):
        raise ValueError("Signals contain missing or infinite samples.")

    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")

    difference = actual - expected
    worst = int(np.argmax(np.abs(difference)))
    rmse = np.sqrt(np.mean(difference**2))
    reference_rms = np.sqrt(np.mean(expected**2))

    return pd.Series({
        "max_absolute_error": np.max(np.abs(difference)),
        "rmse": rmse,
        "relative_rmse_percent": (
            100 * rmse / reference_rms if reference_rms > 0 else np.nan
        ),
        "mean_error": np.mean(difference),
        "worst_sample": worst,
        "worst_time_seconds": worst / sampling_rate,
    })

## 2. FIR

In [11]:
raw = np.asarray(raw_channel.data, dtype=float)
reference = np.asarray(reference_channel_fir.data, dtype=float)

reference_comb = np.asarray(reference_channel_comb.data, dtype=float)
reference_integrate = np.asarray(reference_channel_integrate.data, dtype=float)

### Parameters

In [12]:
sampling_rate = raw_channel.samples_per_second

low_hz = 28.0
high_hz = 500.0
num_taps = 285

fir_scale = False
fir_window = [0.44959, 0.49364, 0.05677, 0.0]
fir_padding = "edge"

In [13]:
assert raw.ndim == reference.ndim == 1, "Use one-dimensional channel arrays."
assert raw.size > 0 and reference.size > 0, "An array is empty."
assert np.isfinite(raw).all(), "Raw signal contains NaN or infinity."
assert np.isfinite(reference).all(), "Reference contains NaN or infinity."
assert np.isfinite(sampling_rate) and sampling_rate > 0, "Check the sampling rate."

print(f"Sampling rate:       {sampling_rate} Hz")
print(f"Raw samples:         {raw.size}")
print(f"Reference samples:   {reference.size}")
print(f"Raw duration:        {raw.size / sampling_rate:.6f} seconds")
print(f"Same length:         {raw.size == reference.size}")
print(f"Nyquist frequency:   {sampling_rate / 2} Hz")

Sampling rate:       2000.0 Hz
Raw samples:         805412
Reference samples:   805412
Raw duration:        402.706000 seconds
Same length:         True
Nyquist frequency:   1000.0 Hz


In [14]:
# Reproduce the calculation in your original function.
calculated_count = 4 * sampling_rate / low_hz
num_taps = int(calculated_count)

# Reproduce its adjustment to an odd number.
if num_taps % 2 == 0:
    num_taps += 1

half_length = (num_taps - 1) // 2

print("Count before rounding:", calculated_count)
print("Count used by your Python function:", num_taps)
print(f"Half-length:            {half_length} samples")
print(f"Half-length in time:    {1000 * half_length / sampling_rate:.2f} ms")

if high_hz >= sampling_rate / 2:
    raise ValueError(
        "This implementation requires a sampling rate greater than "
        "1000 Hz for a 500 Hz upper cutoff."
    )

Count before rounding: 285.7142857142857
Count used by your Python function: 285
Half-length:            142 samples
Half-length in time:    71.00 ms


## Functions

In [15]:
def biopac_fir_bandpass(data, sampling_rate, low_hz=28.0, high_hz=500.0, num_taps=285):
    """
    Approximate the AcqKnowledge FIR band-pass tested in this project.

    Uses a symmetric custom Blackman window, no gain normalization,
    edge padding, and centered convolution.

    The default 285 coefficients were confirmed in AcqKnowledge for
    the tested 2000 Hz recording. For other settings, supply the
    coefficient count shown in AcqKnowledge.

    Returns a new 1D array with the same length and units as data.
    """
    x = np.asarray(data, dtype=float)

    if x.ndim != 1 or x.size == 0:
        raise ValueError("data must be a nonempty, one-dimensional array.")
    if not np.isfinite(x).all():
        raise ValueError("data contains NaN or infinite values.")
    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")
    if not 0 < low_hz < high_hz < sampling_rate / 2:
        raise ValueError(
            "Cutoffs must satisfy: 0 < low_hz < high_hz < sampling_rate / 2."
        )
    if (
        isinstance(num_taps, (bool, np.bool_))
        or not isinstance(num_taps, (int, np.integer))
        or num_taps < 3
        or num_taps % 2 == 0
    ):
        raise ValueError("num_taps must be an odd integer of at least 3.")

    # Design the base FIR without gain normalization.
    coefficients = signal.firwin(
        numtaps=num_taps,
        cutoff=[low_hz, high_hz],
        fs=sampling_rate,
        pass_zero=False,
        window="boxcar",
        scale=False,
    )

    # Apply the custom window used in our comparison.
    window = signal.windows.general_cosine(
        num_taps,
        [0.44959, 0.49364, 0.05677, 0.0],
        sym=True,
    )
    coefficients *= window

    # Repeat endpoint values and center the filtered output.
    half_length = (num_taps - 1) // 2
    padded = np.pad(x, (half_length, half_length), mode="edge")

    return signal.convolve(padded, coefficients, mode="valid")

In [16]:
def comb_band_stop_candidate(
    data,
    sampling_rate,
    base_hz=60.0,
    q=5.0,
):
    """
    Experimental comb band-stop using cascaded RBJ notch filters.

    Applies one forward pass with zero initial state at each harmonic
    strictly below Nyquist. Preserves signal length and units.

    Agreement with AcqKnowledge must be checked against its output.
    """
    filtered = np.asarray(data, dtype=float).copy()

    if filtered.ndim != 1 or filtered.size == 0:
        raise ValueError("data must be a nonempty 1D array.")

    if not np.isfinite(filtered).all():
        raise ValueError("data contains NaN or infinite values.")

    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")

    nyquist = sampling_rate / 2

    if not np.isfinite(base_hz) or not 0 < base_hz < nyquist:
        raise ValueError("base_hz must be positive and below Nyquist.")

    if not np.isfinite(q) or q <= 0:
        raise ValueError("q must be positive and finite.")

    frequencies = np.arange(base_hz, nyquist, base_hz)

    for frequency in frequencies:
        omega = 2 * np.pi * frequency / sampling_rate
        alpha = np.sin(omega) / (2 * q)
        cosine = np.cos(omega)

        b = np.array([1.0, -2 * cosine, 1.0])
        a = np.array([1 + alpha, -2 * cosine, 1 - alpha])

        a0 = a[0]
        b /= a0
        a /= a0

        filtered = signal.lfilter(b, a, filtered)

    return filtered

In [17]:
def comb_band_stop_initialized(
    data,
    sampling_rate,
    base_hz=60.0,
    q=5.0,
):
    """
    Experimental comb band-stop using cascaded RBJ notch filters.

    Applies one forward pass with zero initial state at each harmonic
    strictly below Nyquist. Preserves signal length and units.

    Agreement with AcqKnowledge must be checked against its output.
    """
    filtered = np.asarray(data, dtype=float).copy()

    if filtered.ndim != 1 or filtered.size == 0:
        raise ValueError("data must be a nonempty 1D array.")

    if not np.isfinite(filtered).all():
        raise ValueError("data contains NaN or infinite values.")

    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")

    nyquist = sampling_rate / 2

    if not np.isfinite(base_hz) or not 0 < base_hz < nyquist:
        raise ValueError("base_hz must be positive and below Nyquist.")

    if not np.isfinite(q) or q <= 0:
        raise ValueError("q must be positive and finite.")

    frequencies = np.arange(base_hz, nyquist, base_hz)

    for frequency in frequencies:
        omega = 2 * np.pi * frequency / sampling_rate
        alpha = np.sin(omega) / (2 * q)
        cosine = np.cos(omega)

        b = np.array([1.0, -2 * cosine, 1.0])
        a = np.array([1 + alpha, -2 * cosine, 1 - alpha])

        a0 = a[0]
        b /= a0
        a /= a0

        # Initialize each notch as if its input had been constant
        # at that stage's first value before the recording began.
        zi = signal.lfilter_zi(b, a) * filtered[0]
        filtered, _ = signal.lfilter(b, a, filtered, zi=zi)

    return filtered

In [18]:
def biopac_moving_rms_candidate(data, window_samples=1000):
    """
    Trailing RMS candidate.

    Uses a fixed denominator of window_samples.
    Missing samples before the recording are treated as zero.
    AcqKnowledge equivalence remains to be tested.
    """
    x = np.asarray(data, dtype=float)

    if x.ndim != 1 or x.size == 0:
        raise ValueError("data must be a nonempty 1D array.")
    if not np.isfinite(x).all():
        raise ValueError("data contains NaN or infinite values.")
    if (
        isinstance(window_samples, (bool, np.bool_))
        or not isinstance(window_samples, (int, np.integer))
        or window_samples < 1
    ):
        raise ValueError("window_samples must be a positive integer.")

    window_sums = np.convolve(
        x**2,
        np.ones(window_samples),
        mode="full",
    )[:x.size]

    return np.sqrt(window_sums / window_samples)

In [19]:
def biopac_moving_rms(data, window_samples=1000):
    """
    Approximates AcqKnowledge Transform > Integrate > Root mean square.

    Uses a trailing window of 1000 samples by default, matching our protocol.
    Each output includes the current sample and preceding samples.
    At the beginning, only available samples contribute to the sum.

    Uses a provisional fixed denominator of window_samples - 1,
    including during startup. AcqKnowledge's manual gives conflicting
    descriptions of the denominator.

    Expects a nonempty 1D signal.
    Returns a new array with the same length, leaving the input unchanged.
    Exact AcqKnowledge equivalence requires numerical comparison.
    """
    if window_samples < 2:
        raise ValueError("window_samples must be at least 2.")

    squared = np.asarray(data, dtype=float) ** 2

    # Trailing sums: current sample plus up to window_samples - 1 previous samples.
    # Taking the first len(data) outputs preserves timing and signal length.
    # No reflected samples are introduced at the beginning.
    window_sums = np.convolve( squared, np.ones(window_samples), mode="full")[:len(squared)]

    # Provisional fixed denominator, including for incomplete startup windows.
    # TODO: verify both startup values and s versus s-1 against AcqKnowledge.
    # For a full 1000-sample window of alternating +1/-1:
    # denominator 1000 -> 1.000000; denominator 999 -> 1.000500375...
    mean_squared = window_sums / (window_samples - 1)

    return np.sqrt(mean_squared)

### Testing process

In [20]:
python_filtered = biopac_fir_bandpass(
    raw,
    sampling_rate=2000.0,
    num_taps=285,
)

In [21]:
compare(python_filtered, reference, sampling_rate)

max_absolute_error       1.730032e-04
rmse                     1.220032e-05
relative_rmse_percent    5.785320e-03
mean_error               4.826925e-10
worst_sample             3.877020e+05
worst_time_seconds       1.938510e+02
dtype: float64

In [22]:
python_comb_candidate = comb_band_stop_candidate(
    reference,
    sampling_rate=sampling_rate,
    base_hz=60.0,
    q=5.0,
)

In [23]:
compare(python_comb_candidate, reference_comb, sampling_rate)

max_absolute_error       4.702251e-02
rmse                     1.099289e-04
relative_rmse_percent    4.617764e+00
mean_error               2.847017e-07
worst_sample             0.000000e+00
worst_time_seconds       0.000000e+00
dtype: float64

In [24]:
trim = int(sampling_rate)
trim

2000

In [25]:
compare(
    python_comb_candidate[trim:-trim],
    reference_comb[trim:-trim],
    sampling_rate,
)

max_absolute_error       1.187765e-14
rmse                     1.264909e-15
relative_rmse_percent    6.076999e-11
mean_error               3.832710e-18
worst_sample             1.000000e+01
worst_time_seconds       5.000000e-03
dtype: float64

In [26]:
python_comb_initialized = comb_band_stop_initialized(
    reference,
    sampling_rate=sampling_rate,
    base_hz=60.0,
    q=5.0,
)

print(compare(
    python_comb_initialized,
    reference_comb,
    sampling_rate,
))

max_absolute_error       6.156317e-15
rmse                     1.262769e-15
relative_rmse_percent    5.304493e-11
mean_error               3.775027e-18
worst_sample             6.169530e+05
worst_time_seconds       3.084765e+02
dtype: float64


In [27]:
python_rms = biopac_moving_rms_candidate(
    reference_comb,
    window_samples=1000,
)

print(compare(python_rms, reference_integrate, sampling_rate))

max_absolute_error        0.171136
rmse                      0.001417
relative_rmse_percent    45.894781
mean_error               -0.000033
worst_sample             10.000000
worst_time_seconds        0.005000
dtype: float64


In [28]:
# Exclude the first 1,000 samples (0.5 seconds).
print(compare(
    python_rms[1000:],
    reference_integrate[1000:],
    sampling_rate,
))

max_absolute_error       1.622915e-05
rmse                     1.046412e-06
relative_rmse_percent    5.001251e-02
mean_error              -9.436362e-07
worst_sample             0.000000e+00
worst_time_seconds       0.000000e+00
dtype: float64


In [29]:
python_rms_original = biopac_moving_rms(reference_comb)

In [30]:
print(compare(
    python_rms_original[1000:],
    reference_integrate[1000:],
    sampling_rate,
))

max_absolute_error       4.163336e-17
rmse                     1.001719e-18
relative_rmse_percent    4.787643e-14
mean_error              -5.121579e-20
worst_sample             0.000000e+00
worst_time_seconds       0.000000e+00
dtype: float64


In [31]:
indices = [0, 1, 2, 10, 100, 500, 998, 999, 1000]

print(pd.DataFrame({
    "sample": indices,
    "comb_input": reference_comb[indices],
    "Python_RMS": python_rms_original[indices],
    "AcqKnowledge_RMS": reference_integrate[indices],
}))

   sample  comb_input  Python_RMS  AcqKnowledge_RMS
0       0   -0.073519    0.002326          0.073519
1       1   -0.040656    0.002658          0.084012
2       2    0.008004    0.002670          0.059674
3      10    0.217319    0.019025          0.190151
4     100   -0.021171    0.032113          0.101499
5     500    0.002511    0.032509          0.045951
6     998   -0.001237    0.032533          0.032550
7     999   -0.001844    0.032533          0.032533
8    1000   -0.001913    0.032450          0.032450


In [32]:
def biopac_moving_rms(data, window_samples=1000):
    """
    Trailing RMS matching the AcqKnowledge behavior observed so far.

    Denominator:
        max(available_samples - 1, 1)

    Once the window is full, the denominator is window_samples - 1.
    Returns an array with the same length and units as the input.
    """
    x = np.asarray(data, dtype=float)

    if x.ndim != 1 or x.size == 0:
        raise ValueError("data must be a nonempty 1D array.")
    if not np.isfinite(x).all():
        raise ValueError("data contains NaN or infinite values.")
    if (
        isinstance(window_samples, (bool, np.bool_))
        or not isinstance(window_samples, (int, np.integer))
        or window_samples < 2
    ):
        raise ValueError("window_samples must be an integer of at least 2.")

    window_sums = np.convolve(
        x**2,
        np.ones(window_samples),
        mode="full",
    )[:x.size]

    available_samples = np.minimum(
        np.arange(x.size) + 1,
        window_samples,
    )
    denominator = np.maximum(available_samples - 1, 1)

    return np.sqrt(window_sums / denominator)

In [33]:
python_rms = biopac_moving_rms(reference_comb, window_samples=1000)

print(compare(python_rms, reference_integrate, sampling_rate))

max_absolute_error       4.163336e-17
rmse                     1.197145e-18
relative_rmse_percent    3.876985e-14
mean_error              -3.592891e-20
worst_sample             8.800000e+02
worst_time_seconds       4.400000e-01
dtype: float64


### Final test

In [34]:
python_fir = biopac_fir_bandpass(raw, sampling_rate)

python_comb = comb_band_stop_initialized(
    python_fir, sampling_rate, base_hz=60.0, q=5.0
)

python_rms = biopac_moving_rms(
    python_comb, window_samples=1000
)

In [35]:
rms_reference = np.asarray(
    reference_recording.channels[4].data, dtype=float
)

In [36]:
compare(python_rms, rms_reference, sampling_rate)

max_absolute_error       5.772876e-05
rmse                     4.151517e-07
relative_rmse_percent    1.344480e-02
mean_error              -2.048098e-07
worst_sample             1.000000e+00
worst_time_seconds       5.000000e-04
dtype: float64

In [37]:
skip = 1000  # one RMS window
compare(python_rms[skip:], rms_reference[skip:], sampling_rate)

max_absolute_error       1.687985e-05
rmse                     3.080597e-07
relative_rmse_percent    1.472349e-02
mean_error              -1.990704e-07
worst_sample             3.876690e+05
worst_time_seconds       1.938345e+02
dtype: float64